# Airline AI Assistant - Project

In [48]:
# import necessary libraries
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [49]:
# Load environment variables from .env file

load_dotenv(override=True)

anthropic_base_url = os.getenv("anthropic_url")       # was: anthropic_base_url
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")    # was: anthropic_api_key (lowercase)
MODEL = "claude-sonnet-4-6"                           # was: claude-sonnet-4-5-20250929
openai = OpenAI(base_url=anthropic_base_url, api_key=anthropic_api_key)

In [50]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [51]:
# Define the chat function

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [9]:
gr.ChatInterface(fn=chat, type="messages", title="FlightAI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# Adding Tools to fetch ticket price tool

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.


In [52]:
# Sample data for ticket prices
ticket_prices = {"London": 500, "Paris": 400, "New York": 700, "Tokyo": 800, "Sydney": 900 , "Dubai": 600, "Rome": 450, "Barcelona": 550, "Amsterdam": 480, "Berlin": 520, "Singapore": 750, "Hong Kong": 650, "Los Angeles": 680, "Chicago": 620, "Toronto": 580, "Vancouver": 540, "Mexico City": 470, "Buenos Aires": 530, "Sao Paulo": 610, "Moscow": 490}

In [53]:
# Define the tool function to get ticket prices
def get_ticket_price(destination_city):
    print(f"Tool called with destination: {destination_city}")
    price = ticket_prices.get(destination_city.title(), "Sorry, we don't have ticket prices for that destination.")
    return f"The ticket price to {destination_city} is ${price}."

In [54]:
get_ticket_price("london")

Tool called with destination: london


'The ticket price to london is $500.'

In [55]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [56]:
# List of tools to be passed to the model. Each tool is a dictionary with the keys "type" and "function". The "type" key should have the value "function", and the "function" key should have the value of the function definition dictionary we created above.
tools = [{"type": "function", "function": price_function}]

In [57]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [58]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [59]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [42]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called with destination: London
Tool called with destination: Paris
Tool called with destination: India


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [60]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [61]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [62]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [63]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [64]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


# Using actual DB 

In [65]:
import sqlite3

In [66]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS ticket_prices (city TEXT PRIMARY KEY, price INTEGER)')
    conn.commit()

In [67]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.title(),))
        result = cursor.fetchone()
        if result:
            price = result[0]
            return f"The ticket price to {city} is ${price}."
        else:
            return "Sorry, we don't have ticket prices for that destination."

In [68]:
get_ticket_price("london")

DATABASE TOOL CALLED: Getting price for london


OperationalError: no such table: prices

In [69]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [70]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

OperationalError: no such table: prices

In [47]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


OperationalError: no such table: prices

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()